# GhostTracker — Coveo Pre-train → Fine-tune → HDBSCAN 클러스터링

**파이프라인:**
1. Coveo SIGIR 2021 공개 데이터 → `PAGE|SEMANTIC|CONTEXTUAL` 토큰 변환
2. Transformer MLM pre-train (Coveo)
3. 우리 세션 데이터로 fine-tune
4. [CLS] 임베딩 → UMAP → HDBSCAN 클러스터링
5. 결과 시각화 & 저장

**필요한 파일 (Google Drive에 올려두기):**
- `SIGIR-ecom-data-challenge/train/browsing_train.csv` (Coveo 원본)
- `session_sequences_v2_merged.csv` (우리 세션 데이터)

In [ ]:
# ── 패키지 설치 ──────────────────────────────────────────────────
!pip install -q umap-learn hdbscan scikit-learn

In [ ]:
# ── Google Drive 마운트 ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

BASE = '/content/drive/MyDrive/GhostTracker'
print(f'Drive 마운트 완료. BASE={BASE}')


In [ ]:
# ── 설정 ─────────────────────────────────────────────────────────
import os, torch

# 파일 경로
COVEO_CSV = f'{BASE}/browsing_train.csv'
OUR_CSV   = f'{BASE}/session_sequences_v2_merged.csv'
OUTDIR    = f'{BASE}/output/unsupervised_semantic'
os.makedirs(OUTDIR, exist_ok=True)

# ── 학습 파라미터 (튜닝된 값) ────────────────────────────────────
COVEO_MAX_SESSIONS  = 20000
PRETRAIN_EPOCHS     = 20    # 10 → 20
FINETUNE_EPOCHS     = 60    # 30 → 60
BATCH_SIZE_PRETRAIN = 128
BATCH_SIZE_FINETUNE = 32
LR_PRETRAIN         = 1e-3
LR_FINETUNE         = 5e-4
EMBED_DIM           = 64
MAX_LEN             = 64

# ── 클러스터링 파라미터 (튜닝된 값) ─────────────────────────────
MIN_CLUSTER_SIZE = 8     # 12 → 8  (더 세밀한 클러스터)
MIN_SAMPLES      = 3     # 5  → 3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'OUTDIR: {OUTDIR}')


In [ ]:
# ── Step 1: Coveo → PAGE|SEMANTIC|CONTEXTUAL 변환 ────────────────
import pandas as pd
import csv as csv_mod
from collections import defaultdict
from typing import Dict, List, Tuple
import time

# ── 매핑 테이블 (product_action 기반) ──
EVENT_MAP = {
    ('event_product', 'detail'):   ('PRODUCT',  'VIEW_PRODUCT', 'NONE'),
    ('event_product', 'add'):      ('PRODUCT',  'ADD_CART',     'NONE'),
    ('event_product', 'remove'):   ('CART',     'REMOVE_CART',  'NONE'),
    ('event_product', 'purchase'): ('CHECKOUT', 'CLICK_BUY',    'NONE'),
    ('event_product', 'click'):    ('CATEGORY', 'VIEW_PRODUCT', 'NONE'),  # 목록에서 상품 클릭
    ('event_product', 'promo_click'): ('HOME',  'VIEW_PRODUCT', 'NONE'),
}

# ── Coveo URL 기반 페이지 추론 ──
def infer_page_coveo(url: str) -> str:
    if not isinstance(url, str):
        return 'HOME'
    url = url.lower()
    if any(k in url for k in ('/product', '/item', '/detail', '/dp/')):
        return 'PRODUCT'
    if any(k in url for k in ('/cart', '/basket')):
        return 'CART'
    if any(k in url for k in ('/checkout', '/order', '/payment')):
        return 'CHECKOUT'
    if any(k in url for k in ('/search', '/category', '/collection', '/browse')):
        return 'CATEGORY'
    return 'HOME'

def map_event(etype, paction, url, is_first):
    key = (etype, paction if pd.notna(paction) else '')
    if key in EVENT_MAP:
        return EVENT_MAP[key]
    if etype == 'pageview':
        page = infer_page_coveo(url) if pd.notna(url) else 'HOME'
        if page == 'PRODUCT':
            return (page, 'VIEW_PRODUCT', 'NONE')
        if page == 'CATEGORY':
            return (page, 'ENTER_CATEGORY', 'NONE') if is_first else (page, 'SCROLL_CATEGORY', 'NONE')
        if page == 'CART':
            return (page, 'ENTER_CART', 'NONE')
        if page == 'CHECKOUT':
            return (page, 'ENTER_CHECKOUT', 'NONE')
        # HOME
        return ('HOME', 'ENTER_HOME', 'NONE') if is_first else ('HOME', 'SCROLL_HOME', 'NONE')
    return None

def tok(page, sem, ctx):
    return f'{page}|{sem}|{ctx}'

def build_coveo_sessions(path, max_sessions, min_len=3, max_len=50, chunk_size=500_000):
    print(f'Coveo 로드 중: {path}')
    t0 = time.time()
    buf = defaultdict(list)
    total = 0

    # page_url 컬럼이 있으면 사용, 없으면 생략
    try:
        sample = pd.read_csv(path, nrows=1)
        url_col = 'page_url' if 'page_url' in sample.columns else None
    except Exception:
        url_col = None

    usecols = ['session_id_hash','event_type','product_action','server_timestamp_epoch_ms']
    if url_col:
        usecols.append(url_col)

    reader = pd.read_csv(
        path,
        usecols=usecols,
        dtype={'session_id_hash': str, 'event_type': str, 'product_action': str},
        chunksize=chunk_size,
    )
    for chunk in reader:
        total += len(chunk)
        for _, row in chunk.iterrows():
            buf[row['session_id_hash']].append((
                row['server_timestamp_epoch_ms'],
                row['event_type'],
                row['product_action'],
                row.get(url_col, None) if url_col else None,
            ))
        if max_sessions and len(buf) >= max_sessions * 3:
            break
        if total % 1_000_000 == 0:
            print(f'  {total:,}행 처리 중…')

    print(f'로드 완료: {total:,}행, {len(buf):,} 세션 ({time.time()-t0:.1f}s)')

    sessions = []
    for sid, events in buf.items():
        if max_sessions and len(sessions) >= max_sessions:
            break
        events.sort(key=lambda x: x[0])
        tokens = [tok('HOME', 'START_SESSION', 'NONE')]
        prev_token = None
        for i, (ts, etype, paction, url) in enumerate(events):
            m = map_event(etype, paction, url, i == 0)
            if m:
                t = tok(*m)
                if t != prev_token:   # 연속 중복 제거
                    tokens.append(t)
                    prev_token = t
        if len(tokens) < min_len + 1:  # START_SESSION 제외
            continue
        sessions.append({
            'session_id':      f'coveo_{sid[:16]}',
            'start_timestamp': events[0][0],
            'end_timestamp':   events[-1][0],
            'length':          len(tokens[:max_len]),
            'sequence':        ' '.join(tokens[:max_len]),
        })

    print(f'유효 세션: {len(sessions):,}개')
    return sessions

coveo_sessions = build_coveo_sessions(COVEO_CSV, COVEO_MAX_SESSIONS)

# CSV 저장
coveo_seq_path = f'{OUTDIR}/coveo_session_sequences.csv'
with open(coveo_seq_path, 'w', newline='', encoding='utf-8') as f:
    w = csv_mod.DictWriter(f, fieldnames=['session_id','start_timestamp','end_timestamp','length','sequence'])
    w.writeheader()
    w.writerows(coveo_sessions)
print(f'저장 완료 → {coveo_seq_path}')


In [ ]:
# ── 모델 & 데이터 정의 ───────────────────────────────────────────
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter

PAD_ID, UNK_ID, CLS_ID, MASK_ID = 0, 1, 2, 3
SPECIAL = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[MASK]': 3}

def load_sequences(path, min_len=3):
    sids, raws, toks = [], [], []
    with open(path, newline='', encoding='utf-8-sig') as f:
        for row in csv_mod.DictReader(f):
            s = row.get('sequence', '').strip()
            if not s:
                continue
            t = s.split()
            if len(t) < min_len:
                continue
            sids.append(row.get('session_id', ''))
            raws.append(s)
            toks.append(t)
    return sids, raws, toks

def build_vocab(token_lists):
    c = Counter(t for seq in token_lists for t in seq)
    vocab = dict(SPECIAL)
    for tok_, _ in c.most_common():
        if tok_ not in vocab:
            vocab[tok_] = len(vocab)
    return vocab

def encode(token_lists, vocab, max_len):
    ids = []
    for tokens in token_lists:
        x = [vocab.get(t, UNK_ID) for t in tokens]
        x = x[-max_len:] if len(x) >= max_len else [PAD_ID]*(max_len-len(x)) + x
        ids.append(x)
    return ids

class MaskedDataset(Dataset):
    MASK_PROB = 0.15
    def __init__(self, token_ids):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        ids = self.ids[idx].clone()
        mask = (ids != PAD_ID).long()
        labels = torch.full_like(ids, -100)
        pos = (torch.rand(ids.shape) < self.MASK_PROB) & (mask == 1)
        labels[pos] = ids[pos]
        ids[pos] = MASK_ID
        return ids, mask, labels

class TransformerMLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2,
                 ff_dim=128, max_len=65, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.pos_emb   = nn.Embedding(max_len, embed_dim)
        self.norm      = nn.LayerNorm(embed_dim)
        enc = nn.TransformerEncoderLayer(embed_dim, num_heads, ff_dim, dropout, batch_first=True)
        self.encoder   = nn.TransformerEncoder(enc, num_layers)
        self.head      = nn.Linear(embed_dim, vocab_size)

    def _forward(self, ids, attn_mask):
        B = ids.shape[0]
        cls = torch.full((B,1), CLS_ID, dtype=torch.long, device=ids.device)
        cm  = torch.ones((B,1), dtype=torch.long, device=ids.device)
        ids_c  = torch.cat([cls, ids],  dim=1)
        mask_c = torch.cat([cm, attn_mask], dim=1)
        pos = torch.arange(ids_c.shape[1], device=ids.device).unsqueeze(0)
        x = self.token_emb(ids_c) + self.pos_emb(pos)
        x = self.encoder(x, src_key_padding_mask=(mask_c==0))
        return self.norm(x)

    def forward(self, ids, attn_mask):
        return self.head(self._forward(ids, attn_mask))

    @torch.no_grad()
    def extract_cls(self, ids, attn_mask):
        return self._forward(ids, attn_mask)[:, 0, :].cpu().numpy()

def train_model(model, token_ids, epochs, batch_size, lr, device, label=''):
    loader = DataLoader(MaskedDataset(token_ids), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    model.train()
    for ep in range(1, epochs+1):
        ep_loss = 0
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            logits = model(ids, mask)[:, 1:, :]
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item()
        avg = ep_loss / len(loader)
        losses.append(avg)
        if ep % 5 == 0 or ep == 1:
            print(f'  [{label}] epoch {ep:3d}/{epochs}  loss={avg:.4f}')
    return losses

def extract_embeddings(model, token_ids, device, batch_size=128):
    model.eval()
    ids_t = torch.tensor(token_ids, dtype=torch.long)
    embs = []
    for i in range(0, len(token_ids), batch_size):
        ids = ids_t[i:i+batch_size].to(device)
        embs.append(model.extract_cls(ids, (ids != PAD_ID).long()))
    return np.vstack(embs)

print('모델 정의 완료')

In [ ]:
# ── Step 2: 데이터 로드 & Vocab 구성 ────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

print('우리 데이터 로드…')
our_sids, our_raws, our_toks = load_sequences(OUR_CSV)
print(f'  우리 데이터: {len(our_sids)}세션')

print('Coveo 데이터 로드…')
_, _, coveo_toks = load_sequences(coveo_seq_path)
print(f'  Coveo: {len(coveo_toks)}세션')

# 통합 vocab (우리 + Coveo)
vocab = build_vocab(our_toks + coveo_toks)
print(f'  통합 vocab size: {len(vocab)}')

our_ids    = encode(our_toks,   vocab, MAX_LEN)
coveo_ids  = encode(coveo_toks, vocab, MAX_LEN)

# ── Step 3: Coveo Pre-train ───────────────────────────────────────
model = TransformerMLM(
    vocab_size=len(vocab), embed_dim=EMBED_DIM,
    max_len=MAX_LEN+1
).to(DEVICE)

print(f'\n[Phase 1] Coveo Pre-train ({PRETRAIN_EPOCHS} epochs, {len(coveo_ids)} 세션)…')
t0 = time.time()
pretrain_losses = train_model(model, coveo_ids, PRETRAIN_EPOCHS,
                               BATCH_SIZE_PRETRAIN, LR_PRETRAIN, DEVICE, 'pretrain')
print(f'Pre-train 완료 ({time.time()-t0:.1f}s)')

# 체크포인트 저장
ckpt_path = f'{OUTDIR}/pretrained_model.pt'
torch.save({'model_state': model.state_dict(), 'vocab': vocab, 'vocab_size': len(vocab)}, ckpt_path)
print(f'체크포인트 저장 → {ckpt_path}')

In [ ]:
# ── Step 4: 우리 데이터 Fine-tune ────────────────────────────────
print(f'[Phase 2] Fine-tune ({FINETUNE_EPOCHS} epochs, {len(our_ids)} 세션)…')
t0 = time.time()
# fine-tune은 낮은 LR 사용
for g in model.parameters():
    g.requires_grad_(True)

finetune_losses = train_model(model, our_ids, FINETUNE_EPOCHS,
                               BATCH_SIZE_FINETUNE, LR_FINETUNE, DEVICE, 'finetune')
print(f'Fine-tune 완료 ({time.time()-t0:.1f}s)  최종 loss={finetune_losses[-1]:.4f}')

In [ ]:
# ── Step 5: 임베딩 추출 → UMAP → HDBSCAN ────────────────────────
import hdbscan
import umap
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print('CLS 임베딩 추출…')
embs = extract_embeddings(model, our_ids, DEVICE)
embs_norm = normalize(embs, norm='l2')
print(f'  shape: {embs.shape}')

print('UMAP (64d→10d)…')
reducer = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.1,
                    metric='cosine', random_state=42)
embs_umap = reducer.fit_transform(embs_norm)

print('HDBSCAN 클러스터링…')
clusterer = hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                              min_samples=MIN_SAMPLES, metric='euclidean')
labels = clusterer.fit_predict(embs_umap)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()
N = len(labels)

print(f'\n클러스터 수: {n_clusters}')
print(f'노이즈:      {n_noise} ({100*n_noise/N:.1f}%)')
for c in sorted(set(labels)):
    print(f'  C{c}: {(labels==c).sum()}개')

# 지표
nn = labels != -1
if nn.sum() > 1 and len(set(labels[nn])) > 1:
    sil = silhouette_score(embs_umap[nn], labels[nn])
    db  = davies_bouldin_score(embs_umap[nn], labels[nn])
    ch  = calinski_harabasz_score(embs_umap[nn], labels[nn])
    print(f'\nSilhouette:      {sil:.4f}')
    print(f'Davies-Bouldin:  {db:.4f}')
    print(f'Calinski-Harabasz: {ch:.2f}')

In [ ]:
# ── Step 6: UMAP 2D 시각화 ───────────────────────────────────────
import matplotlib.pyplot as plt

reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=42)
embs_2d = reducer_2d.fit_transform(embs_norm)

unique = sorted(set(labels))
cmap   = plt.cm.get_cmap('tab10', max(len(unique), 1))
colors = {c: 'lightgray' if c == -1 else cmap(i % 10) for i, c in enumerate(unique)}

fig, ax = plt.subplots(figsize=(9, 6))
for c in unique:
    m = labels == c
    ax.scatter(embs_2d[m, 0], embs_2d[m, 1],
               c=[colors[c]], label='Noise' if c==-1 else f'C{c}',
               s=55, alpha=0.75, edgecolors='white', linewidths=0.4)
ax.set_title('Semantic Token Clustering — Coveo Pre-train → Fine-tune (UMAP 2D)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/colab_umap2d.png', dpi=150)
plt.show()
print('시각화 저장 완료')

In [ ]:
# ── Step 7: 클러스터 프로파일 & JSON 저장 ────────────────────────
import json
import numpy as np

# numpy 타입 → Python 기본 타입 변환 (JSON 직렬화 대비)
def _np(v):
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.integer):  return int(v)
    if isinstance(v, np.ndarray):  return v.tolist()
    return v

profiles = []
for c in sorted(set(labels)):
    idx = [i for i, l in enumerate(labels) if l == c]
    sem_cnt = Counter()
    page_cnt = Counter()
    for i in idx:
        for t in our_raws[i].split():
            parts = t.split('|')
            if len(parts) == 3:
                page_cnt[parts[0]] += 1
                sem_cnt[parts[1]]  += 1

    tag = '노이즈' if c == -1 else f'Cluster {c}'
    print(f'\n{"="*50}')
    print(f'{tag}  (n={len(idx)})')
    print('Top actions:', ', '.join(f'{a}({n})' for a, n in sem_cnt.most_common(5)))
    print('Page 분포:', dict(page_cnt.most_common()))

    profiles.append({
        'cluster':     int(c),
        'count':       len(idx),
        'top_actions': [{'action': a, 'count': int(n)} for a, n in sem_cnt.most_common(5)],
        'page_dist':   {p: int(n) for p, n in page_cnt.most_common()},
    })

# numpy 스칼라를 Python float/int로 명시 변환
result = {
    'total_sessions':    int(N),
    'n_clusters':        int(n_clusters),
    'noise_count':       int(n_noise),
    'silhouette':        float(sil),
    'davies_bouldin':    float(db),
    'calinski_harabasz': float(ch),
    'clusters':          profiles,
}

json_path = f'{OUTDIR}/cluster_profiles.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

csv_path = f'{OUTDIR}/semantic_cluster_results.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv_mod.writer(f)
    w.writerow(['session_id', 'cluster', 'umap_x', 'umap_y'])
    for i in range(N):
        w.writerow([our_sids[i], int(labels[i]),
                    round(float(embs_2d[i,0]),4),
                    round(float(embs_2d[i,1]),4)])

print(f'\n완료! 클러스터: {result["n_clusters"]}개 | Silhouette: {result["silhouette"]:.4f} | 총 세션: {result["total_sessions"]}')
print(f'결과 JSON → {json_path}')
print(f'결과 CSV  → {csv_path}')
